# Strategia Content-Based: similarità ibrida testo + metadata
In questa strategia di Content-Based Filtering estendiamo l'approccio classico basato sulla sola similarità testuale, incorporando metadata descrittivi aggiuntivi (categoria e sottocategoria) e il comportamento temporale dell'utente (pesatura per recency). Questa rappresentazione ibrida arricchita permette al sistema di raccomandazione di catturare non solo la similarità semantica tra articoli, ma anche il tipo editoriale, le categorie tematiche e gli interessi più recenti dell'utente, ottenendo un profilo più personalizzato.

L'approccio si articola in tre fasi principali:

## Costruzione della rappresentazione ibrida
Ogni articolo viene codificato con SBERT (titolo + abstract) e arricchito con embedding one-hot di categoria e sottocategoria.

## Profilo utente pesato per recency
Invece di una media uniforme, alle interazioni più recenti dell'utente viene assegnata un'importanza maggiore, producendo un embedding utente più realistico e dinamico.

## Calcolo della similarità (scoring e ranking)
Il vettore del profilo utente viene confrontato con i vettori degli articoli candidati tramite cosine similarity.

## Indice
* [Generazione degli embedding](#embedding-generation)
* [Costruzione della rappresentazione ibrida](#hybrid-feature-construction)
* [Creazione del profilo utente](#user-profile-creation)
* [Scoring e ranking](#scoring-and-ranking)
* [Pipeline di valutazione](#evaluation-pipeline)

In [ ]:

# Import e percorsi

import os
import json
import math
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional, Dict
from collections import Counter, defaultdict
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NEWS_PATH = str(PROJECT_ROOT / "dataset" / "MINDsmall_dev" / "news.tsv")
BEHAV_PATH = str(PROJECT_ROOT / "dataset" / "MINDsmall_dev" / "behaviors.tsv")
EMB_PATH = str(PROJECT_ROOT / "content_based_clicked" / "Embeddings" / "newsembeddings.npy")

In [ ]:
MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:

# Caricamento delle news

news_cols = ['News_ID','Category','SubCategory','Title','Abstract','URL','Title_Entities','Abstract_Entities']
df_news = pd.read_csv(NEWS_PATH, sep='\t', names=news_cols)
df_news['Title'] = df_news['Title'].fillna('')
df_news['Abstract'] = df_news['Abstract'].fillna('')
df_news['full_text'] = df_news['Title'] + " - " + df_news['Abstract']
print(f"Caricati {len(df_news)} articoli")


# Generazione degli embedding
Per creare la base semantica del nostro approccio content-based, calcoliamo prima gli embedding testuali usando un modello Sentence-Transformer. Utilizziamo lo stesso modello ad alte prestazioni della strategia baseline (`all-MiniLM-L6-v2`), che mappa le frasi in uno spazio vettoriale denso a 384 dimensioni.
Questo modello è ottimizzato per task di similarità semantica e offre un ottimo compromesso prestazioni/velocità, essenziale per la raccomandazione di news su larga scala.

Incorporiamo il seguente campo combinato:
* Titolo + " - " + Abstract

Questo garantisce di catturare sia il tema generale sia i segnali contestuali di dettaglio.

**Output**: una matrice di shape (N_articoli × 384) che rappresenta i vettori semantici degli articoli.

In [ ]:
# Embedding SBERT (testo)

if os.path.exists(EMB_PATH):
    print(f"Caricamento embedding SBERT da {EMB_PATH}")
    emb_text = np.load(EMB_PATH)
    if emb_text.shape[0] != len(df_news):
        raise ValueError("Il numero di righe degli embedding non coincide con quello delle news")
else:
    print(f"Calcolo degli embedding SBERT con {MODEL_NAME} ...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)
    emb_text = model.encode(
        df_news['full_text'].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    os.makedirs(os.path.dirname(EMB_PATH), exist_ok=True)
    np.save(EMB_PATH, emb_text)
    print(f"Salvato in {EMB_PATH}")


In [ ]:
# One-hot di categoria/sottocategoria

cat_dummies = pd.get_dummies(df_news['Category'], prefix='cat')
sub_dummies = pd.get_dummies(df_news['SubCategory'], prefix='sub')
cat_mat = cat_dummies.values.astype(np.float32)        # (N, C)
sub_mat = sub_dummies.values.astype(np.float32)        # (N, S)


In [ ]:

# Feature TF-IDF sulle entità
def parse_entities(entity_str: str) -> List[str]:
    if pd.isna(entity_str) or not entity_str.strip():
        return []
    try:
        data = json.loads(entity_str)
        out = []
        for e in data:
            wid = e.get("WikidataId") or e.get("WikidataId".lower())
            if wid:
                out.append(f"WD:{wid}")
            else:
                lbl = e.get("Label")
                if lbl:
                    out.append(f"L:{lbl.strip().lower()}")
        return out
    except Exception:
        return []

entities_list = []
for _, row in df_news[['Title_Entities','Abstract_Entities']].iterrows():
    e1 = parse_entities(row['Title_Entities'])
    e2 = parse_entities(row['Abstract_Entities'])
    entities_list.append(e1 + e2)

# Costruisce il vocabolario
entity_df = []
for ents in entities_list:
    entity_df.extend(ents)
entity_vocab = sorted(set(entity_df))
eid2idx = {e:i for i,e in enumerate(entity_vocab)}

# Term frequency per articolo
rows, cols, vals = [], [], []   # formato COO per costruzione sparsa (alla fine si memorizza denso; N piccolo in MINDsmall)
for i, ents in enumerate(entities_list):
    if not ents: 
        continue
    cnt = Counter(ents)
    for e, c in cnt.items():
        j = eid2idx[e]
        rows.append(i); cols.append(j); vals.append(c)

N = len(df_news)
E = len(entity_vocab)
ent_tf = np.zeros((N, E), dtype=np.float32)
if vals:
    ent_tf[rows, cols] = vals

# IDF
df_counts = (ent_tf > 0).sum(axis=0)  # (E,)
idf = np.log((1 + N) / (1 + df_counts)) + 1.0   # smoothing
ent_tfidf = ent_tf * idf[np.newaxis, :]


ent_tfidf = normalize(ent_tfidf, norm='l2', axis=1, copy=False)

print(f"Dimensione del vocabolario di entità: {E}")


# Costruzione della rappresentazione ibrida
Sebbene gli embedding testuali puri catturino il significato semantico, spesso non colgono segnali editoriali strutturali come la categoria dell'articolo ("Sports", "Health", ecc.). Per questo estendiamo ogni vettore articolo concatenando:

* embedding testuale SBERT
* vettore one-hot della categoria
* vettore one-hot della sottocategoria

Formalmente:
`articolo_ibrido = [SBERT(Titolo, Abstract), OneHot(Categoria), OneHot(Sottocategoria)]`

Dopo la concatenazione applichiamo la normalizzazione L2, in modo che la cosine similarity resti significativa su tutte le dimensioni della rappresentazione.

In [ ]:

# Costruzione dei vettori ibridi degli articoli

w_text = 1.0
w_cat  = 0.3
w_sub  = 0.3
w_ent  = 0.7


# Normalizza per prima cosa gli embedding testuali
text_norms = np.linalg.norm(emb_text, axis=1, keepdims=True)
text_norms[text_norms == 0] = 1.0
emb_text_norm = emb_text / text_norms

# Normalizza riga per riga i blocchi di categoria/sottocategoria
cat_mat_n = normalize(cat_mat, norm='l2', axis=1)
sub_mat_n = normalize(sub_mat, norm='l2', axis=1)

# Concatenazione pesata
hybrid = np.hstack([
    w_text * emb_text_norm,
    w_cat  * cat_mat_n,
    w_sub  * sub_mat_n,
    w_ent  * ent_tfidf
]).astype(np.float32)

# Normalizzazione L2 finale sul vettore concatenato (fondamentale per la cosine similarity)
hyb_norms = np.linalg.norm(hybrid, axis=1, keepdims=True)
hyb_norms[hyb_norms == 0] = 1.0
hyb = hybrid / hyb_norms

print("Shape del vettore ibrido:", hyb.shape)



In [ ]:

# Mapping, comportamenti utente, popolarità

news_id_to_idx = {nid: i for i, nid in enumerate(df_news['News_ID'])}
idx_to_news_id = {i: nid for i, nid in enumerate(df_news['News_ID'])}

beh_cols = ['Impression_ID','User_ID','Time','History','Impressions']
df_beh = pd.read_csv(BEHAV_PATH, sep='\t', names=beh_cols)
df_beh['History'] = df_beh['History'].fillna('')
print(f"Caricate {len(df_beh)} impression")

all_hist = ' '.join(df_beh['History'].tolist()).split()
item_counts = Counter(all_hist)
total_interactions = max(1, len(all_hist))
item_pop = {nid: cnt / total_interactions for nid, cnt in item_counts.items()}
MIN_POP = 1e-6

def novelty_at_k(ids: List[str], k=5) -> float:
    s = 0.0
    for nid in ids[:k]:
        p = item_pop.get(nid, MIN_POP)
        s += -math.log2(p)
    return s / k if k > 0 else 0.0

# Creazione del profilo utente
A differenza della strategia baseline, che utilizza una semplice media non pesata degli embedding di tutti gli articoli cliccati, qui introduciamo una pesatura per recency che riflette l'evoluzione temporale degli interessi dell'utente.

Ai click più recenti viene assegnato un peso esponenzialmente maggiore:

- $i$ = posizione nella cronologia (dalla più vecchia alla più recente),
- $decay \in (0,1)$, tipicamente tra 0.7 e 0.9.

Questo produce un embedding utente che riflette meglio gli interessi attuali, invece del rumore accumulato nella storia a lungo termine.

In [ ]:

# Profilo utente (pesato per recency)

def recency_weights(n_items: int, decay: float = 0.8) -> np.ndarray:
    pos = np.arange(n_items)
    exps = (n_items - 1) - pos
    return (decay ** exps).astype(np.float32)

def build_user_profile(history_ids: List[str], decay: Optional[float]=0.8) -> Optional[np.ndarray]:
    idxs = [news_id_to_idx[nid] for nid in history_ids if nid in news_id_to_idx]
    if not idxs:
        return None
    vecs = hyb[idxs]           # già normalizzati
    if decay is None:
        prof = vecs.mean(axis=0)
    else:
        w = recency_weights(len(vecs), decay=decay)
        w = w / (w.sum() + 1e-12)
        prof = (vecs * w[:, None]).sum(axis=0)
    # normalizza il profilo
    n = np.linalg.norm(prof) or 1.0
    return (prof / n).reshape(1, -1)

# Scoring e ranking
Per ciascuna sessione (impression):

* si costruisce il profilo utente pesato per recency;
* ogni articolo candidato viene rappresentato tramite l'embedding ibrido;
* si calcola la cosine similarity tra il profilo utente e ciascun candidato.

I top-K articoli con punteggio più alto vengono restituiti come raccomandazioni.

In [ ]:

# Scoring + re-ranking MMR

def score_candidates(user_profile: np.ndarray, cand_ids: List[str]) -> Tuple[List[str], np.ndarray]:
    valid, vecs = [], []
    for nid in cand_ids:
        if nid in news_id_to_idx:
            valid.append(nid)
            vecs.append(hyb[news_id_to_idx[nid]])
    if not vecs:
        return [], np.array([])
    V = np.vstack(vecs)                   # (C, D)
    scores = (V @ user_profile.T).ravel() # cosine tramite dot product (vettori già normalizzati)
    return valid, scores

def mmr_rerank(candidate_ids: List[str],
               candidate_scores: np.ndarray,
               top_k: int = 10,
               lamb: float = 0.75) -> List[str]:
    """
    Re-ranking tramite MMR (Maximal Marginal Relevance, consapevole della diversità).
    candidate_ids e candidate_scores sono allineati.
    """
    if len(candidate_ids) <= top_k:
        return candidate_ids

    # Pre-carica i vettori
    V = np.vstack([hyb[news_id_to_idx[nid]] for nid in candidate_ids])  # (C, D)
    # Parte dal documento con score più alto
    order = np.argsort(candidate_scores)[::-1]
    selected = [order[0]]
    selected_set = {selected[0]}

    while len(selected) < min(top_k, len(candidate_ids)):
        best_j = None
        best_val = -1e9
        for j in order:
            if j in selected_set:
                continue
            # rilevanza rispetto all'utente
            rel = candidate_scores[j]
            # penalità di diversità rispetto a quanto già selezionato
            div = 0.0
            for s in selected:
                div = max(div, float(V[j] @ V[s]))
            mmr_val = lamb * rel - (1 - lamb) * div
            if mmr_val > best_val:
                best_val = mmr_val
                best_j = j
        selected.append(best_j)
        selected_set.add(best_j)

    return [candidate_ids[j] for j in selected]

# Pipeline di valutazione
Per valutare le prestazioni di questa strategia content-based ibrida, seguiamo il protocollo di valutazione MIND:

1. costruzione del profilo utente usando solo la `History` passata dell'utente (nessun leakage);
2. estrazione degli articoli candidati dalla `Impression` corrente;
3. calcolo della cosine similarity per tutti i candidati;
4. calcolo delle metriche per ciascuna impression:

   1. AUC
   2. MRR
   3. nDCG@5 / nDCG@10
   4. Precision@5
   5. Recall@5
   6. Novelty@5

In [ ]:

# Metriche

def custom_ndcg_score(y_true, y_score, k=5):
    """
    nDCG@K per etichette binarie (implementazione locale).
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    # Ordina le etichette in base agli score predetti
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order][:k]

    # DCG
    discounts = np.log2(np.arange(2, len(y_true_sorted) + 2))
    dcg = np.sum(y_true_sorted / discounts)

    # DCG ideale (IDCG)
    ideal_sorted = np.sort(y_true)[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_sorted) + 2))
    idcg = np.sum(ideal_sorted / ideal_discounts)

    if idcg == 0.0:
        return 0.0

    return float(dcg / idcg)

def mrr_at_scores(y_true: List[int], y_score: np.ndarray) -> float:
    order = np.argsort(y_score)[::-1]
    y_sorted = np.asarray(y_true)[order]
    for i, lab in enumerate(y_sorted):
        if lab == 1:
            return 1.0 / (i + 1)
    return 0.0

def precision_recall_at_k(y_true: List[int], y_score: np.ndarray, k=5) -> Tuple[float, float]:
    order = np.argsort(y_score)[::-1]
    y_sorted = np.asarray(y_true)[order]
    topk = y_sorted[:k]
    hits = int(topk.sum())
    total_rel = int(np.sum(y_true))
    p = hits / k
    r = hits / total_rel if total_rel > 0 else 0.0
    return p, r


# Valutazione (con e senza MMR)

def evaluate(df_beh: pd.DataFrame, use_mmr: bool = True, lamb: float = 0.75, k_div: int = 10, decay: float = 0.8):
    all_auc, all_mrr, all_ndcg5, all_ndcg10 = [], [], [], []
    all_p5, all_r5, all_nov5 = [], [], []

    for _, row in tqdm(df_beh.iterrows(), total=len(df_beh), desc="Valutazione"):
        history = row['History'].split()
        impressions = str(row['Impressions']).split()
        if not history or not impressions:
            continue

        user_prof = build_user_profile(history, decay=decay)
        if user_prof is None:
            continue

        cand_ids = [c.split('-')[0] for c in impressions]
        y_true = [int(c.split('-')[1]) for c in impressions]
        pos = sum(y_true)
        if pos == 0 or pos == len(y_true):
            continue

        valid_ids, y_score = score_candidates(user_prof, cand_ids)

        # riallinea y_true se alcuni candidati risultano mancanti
        if len(valid_ids) != len(cand_ids):
            keep = set(valid_ids)
            y_true = [yt for nid, yt in zip(cand_ids, y_true) if nid in keep]

        if len(y_true) == 0:
            continue

        # Metriche di base (calcolate PRIMA del re-ranking MMR)
        try:
            all_auc.append(roc_auc_score(y_true, y_score))
        except ValueError:
            pass

        all_mrr.append(mrr_at_scores(y_true, y_score))
        all_ndcg5.append(custom_ndcg_score(y_true, y_score, k=5))
        all_ndcg10.append(custom_ndcg_score(y_true, y_score, k=10))
        p5, r5 = precision_recall_at_k(y_true, y_score, k=5)

        # Lista di raccomandazione diversificata (MMR) usata solo per la Novelty@5
        order = np.argsort(y_score)[::-1]
        ranked_ids = [valid_ids[i] for i in order]

        if use_mmr:
            reranked_ids = mmr_rerank(ranked_ids, y_score[order], top_k=10, lamb=lamb)
            all_nov5.append(novelty_at_k(reranked_ids, k=5))
        else:
            all_nov5.append(novelty_at_k(ranked_ids, k=5))

    return {
        "AUC":        float(np.mean(all_auc)) if all_auc else float('nan'),
        "MRR":        float(np.mean(all_mrr)) if all_mrr else float('nan'),
        "nDCG@5":     float(np.mean(all_ndcg5)) if all_ndcg5 else float('nan'),
        "nDCG@10":    float(np.mean(all_ndcg10)) if all_ndcg10 else float('nan'),
        "Precision@5":float(np.mean(all_p5)) if all_p5 else float('nan'),
        "Recall@5":   float(np.mean(all_r5)) if all_r5 else float('nan'),
        "Novelty@5":  float(np.mean(all_nov5)) if all_nov5 else float('nan'),
    }


In [ ]:

# ===== Esecuzione della valutazione =====
metrics_hybrid_mmr = evaluate(df_beh, use_mmr=True, lamb=0.75, k_div=10, decay=0.8)
print("\n--- IBRIDO + RECENCY + MMR ---")
for k, v in metrics_hybrid_mmr.items():
    print(f"{k:12s} {v:.4f}")

metrics_hybrid_nommr = evaluate(df_beh, use_mmr=False, decay=0.8)
print("\n--- IBRIDO + RECENCY (senza MMR) ---")
for k, v in metrics_hybrid_nommr.items():
    print(f"{k:12s} {v:.4f}")